---
title: Popen - Advanced Usage of subprocess
created: '2019-10-19T00:00:00-07:00'
date: '2026-11-22T17:41:07-08:00'
authors:
  - bendu
label: popen-advanced-usage-of-subprocess
license: CC-BY-4.0
tags:
  - programming
  - Python
  - subprocess
  - Popen
  - adavanced usage
---

**Things on this page are fragmentary and immature notes/thoughts of the author. Please read with your own judgement!**


## Popen - Advance Usage of subprocess

Capture and handle the output of a long running process.

In [3]:
import subprocess as sp
import shlex
import time

In [4]:
def run_command(cmd):
    process = sp.Popen(shlex.split(cmd), stdout=sp.PIPE)
    while True:
        output = process.stdout.readline().decode()
        if output == "" and process.poll() is not None:
            break
        if output:
            print(output.strip())
            print("----")
        time.sleep(0.1)
    rc = process.poll()
    return rc

In [8]:
process = sp.Popen("date; date; date; date;", shell=True, stdout=sp.PIPE)

In [9]:
process.stdout.readline().decode()

'Sat 24 Aug 2019 03:23:36 PM PDT\n'

In [10]:
process.stdout.readline().decode()

'Sat 24 Aug 2019 03:23:36 PM PDT\n'

In [11]:
process.stdout.readline().decode()

'Sat 24 Aug 2019 03:23:36 PM PDT\n'

In [12]:
process.stdout.readline().decode()

'Sat 24 Aug 2019 03:23:36 PM PDT\n'

In [13]:
process.stdout.readline().decode()

''

In [14]:
process.poll()

0

https://www.endpoint.com/blog/2015/01/28/getting-realtime-output-using-python

```
import subprocess
import shlex
import time

def run_command(cmd):
    process = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE)
    while True:
        output = process.stdout.readline().decode()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
            print('----')
        time.sleep(1)
    rc = process.poll()
    return rc
```

## Read the Output of a Subprocess in Real Time

The recommended way to read the output of a long-running process line by line in real time
is to iterate over `process.stdout` directly.

```python
import subprocess as sp


def run_command(cmd: list[str]) -> int:
    with sp.Popen(
        cmd, stdout=sp.PIPE, stderr=sp.STDOUT, text=True, bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="")
    return process.returncode
```

`for line in iter(process.stdout.readline, ""):` is an equivalent (older) idiom.
Notice that `bufsize=1` only sets line buffering on the parent side
and does not change how the subprocess buffers its output (see tip 5 below).
With `text=True`,
undecodable output raises `UnicodeDecodeError` in the middle of the stream.
Pass `errors="replace"` (or a proper `encoding`) when running arbitrary commands.

Below are some tips and clarifications on reading output in real time.

1. `readline()` returns an empty string `""` **only** at EOF
   (i.e., after the subprocess and any other process holding the pipe have closed their write ends of the pipe).
   If the subprocess starts background processes which inherit its stdout,
   the loop waits until they exit too.
   An empty line in the output is returned as `"\n"`, not `""`.
   So, empty lines in the middle of the output are NOT lost
   and are not confused with the end of the output.
   This also means that it is safe to stop reading when `readline()` returns `""`
   without checking `process.poll()`.

2. You do not lose output if the subprocess finishes early.
   Output written by the subprocess stays in the pipe after it exits
   and `readline()` keeps returning it until EOF.
   The loop from the endpoint.com article above is correct
   because it keeps reading until `readline()` returns `""`;
   its extra `process.poll()` check is redundant.
   Checking `process.poll()` alone (without draining the pipe) might lose output.

3. There is no need to call `time.sleep` in the loop.
   `readline()` blocks until a line is available,
   so sleeping only delays output.
   For example,
   the loop from the endpoint.com article calls `time.sleep(1)` once per line
   and thus reads at most about 1 line per second.
   If the subprocess writes faster than you read,
   the pipe buffer fills up and the subprocess blocks on writing (which slows it down),
   although no output is lost.

4. If you set both `stdout=sp.PIPE` and `stderr=sp.PIPE`
   but read only one of them,
   the subprocess might deadlock when the other pipe buffer fills up.
   Either merge stderr into stdout (`stderr=sp.STDOUT`),
   read the two pipes in separate threads,
   or use `process.communicate()` if you do not need real time output.

5. If output arrives in big chunks instead of line by line,
   it is most likely the subprocess buffering its output
   (many programs switch from line buffering to block buffering
   when stdout is a pipe rather than a terminal).
   Disable buffering in the subprocess,
   e.g., `python -u`, the environment variable `PYTHONUNBUFFERED=1`,
   or `stdbuf -oL <command>` for programs using C stdio.

For more discussions,
please refer to
[Getting realtime output using subprocess](https://stackoverflow.com/questions/803265/getting-realtime-output-using-subprocess),
[Read subprocess stdout line by line](https://stackoverflow.com/questions/2804543/read-subprocess-stdout-line-by-line)
and
[Non-blocking read on a subprocess.PIPE in Python](https://stackoverflow.com/questions/375427/non-blocking-read-on-a-subprocess-pipe-in-python).

In [ ]:
## Terminate a Python subprocess launched with `shell=True`

https://stackoverflow.com/questions/4789837/how-to-terminate-a-python-subprocess-launched-with-shell-true

## References 

https://docs.python.org/3/library/subprocess.html#subprocess.Popen

https://www.endpoint.com/blog/2015/01/28/getting-realtime-output-using-python

https://codecalamity.com/run-subprocess-run/

https://stackoverflow.com/questions/53209127/subprocess-unexpected-keyword-argument-capture-output

## References

[Hands on the Python module subprocess](hands-on-the-python-module-subprocess)